In [1]:
import duckdb
import pandas as pd

# 1. Initialize an in-memory DuckDB connection
con = duckdb.connect()

# 2. Define the path to your raw data
data_path = r'E:/freelancing/saas-churn-prediction-pipeline/data/raw/'

# 3. List all the files you downloaded
files = [
    'train.csv', 'train_v2.csv',
    'transactions.csv', 'transactions_v2.csv',
    'user_logs.csv', 'user_logs_v2.csv'
]

# 4. Loop through and extract just the first 3 rows and the column names
for file in files:
    print(f"=========================================")
    print(f"Schema & Preview for: {file}")
    print(f"=========================================")
    
    try:
        # read_csv_auto smartly infers columns. LIMIT 3 ensures we don't crash the RAM.
        query = f"SELECT * FROM read_csv_auto('{data_path}{file}') LIMIT 3"
        
        # Convert the tiny result to a Pandas dataframe just for pretty printing
        df_preview = con.execute(query).df()
        
        print(df_preview.to_string(index=False))
        print("\n")
        
    except Exception as e:
        print(f"Error reading {file}. Check if the file is extracted and the path is correct.")
        print(f"Error detail: {e}\n")

Schema & Preview for: train.csv
                                        msno  is_churn
waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=         1
QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=         1
fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=         1


Schema & Preview for: train_v2.csv
                                        msno  is_churn
ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=         1
f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=         1
zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=         1


Schema & Preview for: transactions.csv
                                        msno  payment_method_id  payment_plan_days  plan_list_price  actual_amount_paid  is_auto_renew  transaction_date  membership_expire_date  is_cancel
YyO+tlZtAXYXoZhNr3Vg3+dfVQvrBVGO8j1mfqe4ZHc=                 41                 30              129                 129              1          20150930                20151101          0
AZtu6Wl0gPojrEQYB8Q3vBSmE2wnZ3hi1FbK1rQQ0A4=                 41           

In [4]:
import duckdb
import time

print("Initializing DuckDB connection and mapping source files...")

# Initialize connection
con = duckdb.connect()
data_path = '../data/raw/'

# Map all required CSV files to virtual views
# This does not load them into memory, it simply registers the schema
try:
    con.execute(f"CREATE VIEW train AS SELECT * FROM read_csv_auto('{data_path}train.csv')")
    con.execute(f"CREATE VIEW train_v2 AS SELECT * FROM read_csv_auto('{data_path}train_v2.csv')")
    
    con.execute(f"CREATE VIEW transactions AS SELECT * FROM read_csv_auto('{data_path}transactions.csv')")
    con.execute(f"CREATE VIEW transactions_v2 AS SELECT * FROM read_csv_auto('{data_path}transactions_v2.csv')")
    
    con.execute(f"CREATE VIEW user_logs AS SELECT * FROM read_csv_auto('{data_path}user_logs.csv')")
    con.execute(f"CREATE VIEW user_logs_v2 AS SELECT * FROM read_csv_auto('{data_path}user_logs_v2.csv')")
    
    print("Source files successfully mapped.")
except Exception as e:
    print(f"Failed to map source files. Verify file paths. Error: {e}")
    raise

print("Executing ETL query...")
start_time = time.time()

# Define the extraction and aggregation logic using CTEs
query = """
WITH all_transactions AS (
    SELECT * FROM transactions
    UNION ALL
    SELECT * FROM transactions_v2
),
all_logs AS (
    SELECT * FROM user_logs
    UNION ALL
    SELECT * FROM user_logs_v2
),
agg_transactions AS (
    SELECT 
        msno,
        COUNT(transaction_date) as total_transactions,
        SUM(actual_amount_paid) as total_revenue,
        MAX(is_auto_renew) as currently_auto_renews,
        MAX(is_cancel) as has_cancelled_before
    FROM all_transactions
    GROUP BY msno
),
agg_logs AS (
    SELECT 
        msno,
        COUNT(date) as total_active_days,
        SUM(num_25) as total_songs_skipped,
        SUM(num_100) as total_songs_completed,
        SUM(total_secs) as total_listen_time_secs
    FROM all_logs
    GROUP BY msno
)
SELECT 
    t.msno,
    t.is_churn,
    trans.total_transactions,
    trans.total_revenue,
    trans.currently_auto_renews,
    trans.has_cancelled_before,
    logs.total_active_days,
    logs.total_songs_skipped,
    logs.total_songs_completed,
    logs.total_listen_time_secs
FROM train_v2 t
LEFT JOIN agg_transactions trans ON t.msno = trans.msno
LEFT JOIN agg_logs logs ON t.msno = logs.msno
"""

output_path = '../data/processed/master_churn_data.parquet'

# Execute aggregation and write directly to disk
try:
    con.execute(f"COPY ({query}) TO '{output_path}' (FORMAT PARQUET)")
    
    duration = time.time() - start_time
    print("Data engineering complete.")
    print(f"Saved directly to Parquet in {duration:.2f} seconds.")
    
    # Verify the output
    result = con.execute(f"SELECT COUNT(*) FROM read_parquet('{output_path}')").fetchone()
    print(f"Total Unique Users Processed: {result[0]}")
    
except Exception as e:
    print(f"Query execution failed: {e}")

Initializing DuckDB connection and mapping source files...
Source files successfully mapped.
Executing ETL query...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data engineering complete.
Saved directly to Parquet in 789.81 seconds.
Total Unique Users Processed: 970960
